In [1]:
import os
from cognite.client import CogniteClient, ClientConfig
from cognite.client.credentials import OAuthClientCredentials

In [ ]:
BASE_URL = "https://aw-was-gp-001.cognitedata.com"
PROJECT = "alimentospolarcomercialca"
TOKEN_URL = "https://datamosaix-prod.us.auth0.com/oauth/token"

# 1. OAuth Credentials configured for Auth0
creds = OAuthClientCredentials(
    token_url=TOKEN_URL,
    client_id=os.getenv("CLIENT_ID", "xxx"),
    client_secret=os.getenv("CLIENT_SECRET", "xxx"),
    audience="https://cognitedata.com",  # Required global audience for DataMosaix
    scopes=[]
)

In [3]:
# 2. Client Config
cnf = ClientConfig(
    client_name="superenvases-data-mosaix",
    project=PROJECT,
    credentials=creds,
    base_url=BASE_URL
)

In [4]:
# 3. Instantiate and Test Connection
try:
    client = CogniteClient(cnf)
    spaces = client.data_modeling.spaces.list()
    print("Successfully authenticated via Auth0!")
    print(f"Active spaces found in '{PROJECT}': {[s.space for s in spaces]}")
except Exception as e:
    print("Authentication error:")
    print(e)

Successfully authenticated via Auth0!
Active spaces found in 'alimentospolarcomercialca': ['CommentInstanceSpace', 'FDM_Project_Space', 'IndustrialCanvasInstanceSpace', 'TestSpace', 'scene']


In [5]:
client = CogniteClient(cnf)

In [6]:
from cognite.client.data_classes.data_modeling import SpaceApply

# Assuming your CogniteClient is already instantiated as `client`
my_space = SpaceApply(
    space="TestSpace",
    description="Sandbox space for practicing data modeling",
)

created_space = client.data_modeling.spaces.apply(my_space)
print(f"Space created: {created_space.space}")

Space created: TestSpace


In [7]:
from cognite.client.data_classes.data_modeling import (
    SpaceApply,
    ContainerApply,
    ContainerProperty,
    Text,
    Float64,
    ContainerId,
    ViewApply,
    MappedPropertyApply,
    DataModelApply,
    ViewId,
)

SPACE = "TestSpace"
MODEL_NAME = "PlantProductionModel"
MODEL_VERSION = "v1"

# ---------------------------------------------------------
# 1. ENSURE SPACE EXISTS
# ---------------------------------------------------------
client.data_modeling.spaces.apply(
    SpaceApply(space=SPACE, description="Sandbox space for practicing data modeling")
)


,value
space,TestSpace
description,Sandbox space for practicing data modeling
is_global,False
last_updated_time,2026-07-26 00:05:27.184000
created_time,2026-07-26 00:05:27.184000


In [9]:
# ---------------------------------------------------------
# 2. CREATE CONTAINERS (Physical Storage)
# ---------------------------------------------------------
containers = [
    # Plant Container
    ContainerApply(
        space=SPACE,
        external_id="PlantContainer",
        name="Plant Storage",
        properties={
            "name": ContainerProperty(type=Text()),
            "site_code": ContainerProperty(type=Text()),
        },
    ),
    # Line Container
    ContainerApply(
        space=SPACE,
        external_id="LineContainer",
        name="Line Storage",
        properties={
            "name": ContainerProperty(type=Text()),
            "line_code": ContainerProperty(type=Text()),
        },
    ),
    # Printer Container
    ContainerApply(
        space=SPACE,
        external_id="PrinterContainer",
        name="Printer Storage",
        properties={
            "name": ContainerProperty(type=Text()),
            "printer_code": ContainerProperty(type=Text()),
        },
    ),
    # Production Report Container (Header)
    ContainerApply(
        space=SPACE,
        external_id="ProductionReportContainer",
        name="Production Report Header Storage",
        properties={
            "report_date": ContainerProperty(type=Text()),
            "shift": ContainerProperty(type=Text()),  # "5AM-5PM" or "5PM-5AM"
            "mechanic": ContainerProperty(type=Text()),
            "group": ContainerProperty(type=Text()),
            "label": ContainerProperty(type=Text()),
        },
    ),
    # Production Entry Container (Hourly details)
    ContainerApply(
        space=SPACE,
        external_id="ProductionEntryContainer",
        name="Hourly Production Entry Storage",
        properties={
            "hour_interval": ContainerProperty(type=Text()),
            "hourly_production": ContainerProperty(type=Float64()),
            "cumulative_production": ContainerProperty(type=Float64()),
            "hourly_rework": ContainerProperty(type=Float64()),
            "cumulative_rework": ContainerProperty(type=Float64()),
            "blow_off": ContainerProperty(type=Float64()),
            "efficiency": ContainerProperty(type=Float64()),
            "downtime_minutes": ContainerProperty(type=Float64()),
            "observations": ContainerProperty(type=Text()),
        },
    ),
]

# 1. Dump containers to dictionary format (camelCase)
container_dicts = [c.dump(camel_case=True) for c in containers]

# 2. Strip out 'constraintState' and 'indexState' fields
for c in container_dicts:
    for prop in c.get("properties", {}).values():
        prop.pop("constraintState", None)
        prop.pop("indexState", None)

# 3. Post the cleaned dictionary payload directly
res = client.post(
    f"/api/v1/projects/{client.config.project}/models/containers",
    json={"items": container_dicts}
)
print("Containers created successfully!")


Containers created successfully!


In [11]:
MODEL_VERSION = "v1"

# 1. Define Views with explicit version "v1"
views = [
    ViewApply(
        space=SPACE,
        external_id="Plant",
        version=MODEL_VERSION,
        name="Plant View",
        properties={
            "name": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="PlantContainer"),
                container_property_identifier="name",
            ),
            "site_code": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="PlantContainer"),
                container_property_identifier="site_code",
            ),
        },
    ),
    ViewApply(
        space=SPACE,
        external_id="Line",
        version=MODEL_VERSION,
        name="Line View",
        properties={
            "name": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="LineContainer"),
                container_property_identifier="name",
            ),
            "line_code": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="LineContainer"),
                container_property_identifier="line_code",
            ),
        },
    ),
    ViewApply(
        space=SPACE,
        external_id="Printer",
        version=MODEL_VERSION,
        name="Printer View",
        properties={
            "name": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="PrinterContainer"),
                container_property_identifier="name",
            ),
            "printer_code": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="PrinterContainer"),
                container_property_identifier="printer_code",
            ),
        },
    ),
    ViewApply(
        space=SPACE,
        external_id="ProductionReport",
        version=MODEL_VERSION,
        name="Production Report View",
        properties={
            "report_date": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionReportContainer"),
                container_property_identifier="report_date",
            ),
            "shift": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionReportContainer"),
                container_property_identifier="shift",
            ),
            "mechanic": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionReportContainer"),
                container_property_identifier="mechanic",
            ),
            "group": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionReportContainer"),
                container_property_identifier="group",
            ),
            "label": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionReportContainer"),
                container_property_identifier="label",
            ),
        },
    ),
    ViewApply(
        space=SPACE,
        external_id="ProductionEntry",
        version=MODEL_VERSION,
        name="Production Entry View",
        properties={
            "hour_interval": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="hour_interval",
            ),
            "hourly_production": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="hourly_production",
            ),
            "cumulative_production": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="cumulative_production",
            ),
            "hourly_rework": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="hourly_rework",
            ),
            "cumulative_rework": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="cumulative_rework",
            ),
            "blow_off": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="blow_off",
            ),
            "efficiency": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="efficiency",
            ),
            "downtime_minutes": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="downtime_minutes",
            ),
            "observations": MappedPropertyApply(
                container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
                container_property_identifier="observations",
            ),
        },
    ),
]

client.data_modeling.views.apply(views)

# 2. Bundle views into the Data Model
data_model = DataModelApply(
    space=SPACE,
    external_id="PlantProductionModel",
    version=MODEL_VERSION,
    views=[
        ViewId(space=SPACE, external_id="Plant", version=MODEL_VERSION),
        ViewId(space=SPACE, external_id="Line", version=MODEL_VERSION),
        ViewId(space=SPACE, external_id="Printer", version=MODEL_VERSION),
        ViewId(space=SPACE, external_id="ProductionReport", version=MODEL_VERSION),
        ViewId(space=SPACE, external_id="ProductionEntry", version=MODEL_VERSION),
    ],
)

client.data_modeling.data_models.apply(data_model)
print("Views and Data Model applied successfully!")

Views and Data Model applied successfully!


In [15]:
from cognite.client.data_classes.data_modeling import (
    NodeApply,
    EdgeApply,
    NodeOrEdgeData,
    DirectRelationReference,
    ViewId,
)

SPACE = "TestSpace"
MODEL_VERSION = "v1"

# Reference View IDs
plant_view = ViewId(space=SPACE, external_id="Plant", version=MODEL_VERSION)
line_view = ViewId(space=SPACE, external_id="Line", version=MODEL_VERSION)
printer_view = ViewId(space=SPACE, external_id="Printer", version=MODEL_VERSION)
report_view = ViewId(space=SPACE, external_id="ProductionReport", version=MODEL_VERSION)
entry_view = ViewId(space=SPACE, external_id="ProductionEntry", version=MODEL_VERSION)

# ---------------------------------------------------------
# 1. CREATE STATIC ASSETS (Plant, Lines, Printers)
# ---------------------------------------------------------
nodes = [
    # Plant Node
    NodeApply(
        space=SPACE,
        external_id="plant_main",
        sources=[
            NodeOrEdgeData(
                source=plant_view,
                properties={"name": "Main Manufacturing Plant", "site_code": "PLANT_01"}
            )
        ],
    ),
    # Line Nodes
    NodeApply(
        space=SPACE,
        external_id="line_l1",
        sources=[
            NodeOrEdgeData(
                source=line_view,
                properties={"name": "Line 1", "line_code": "L1"}
            )
        ],
    ),
    NodeApply(
        space=SPACE,
        external_id="line_l3",
        sources=[
            NodeOrEdgeData(
                source=line_view,
                properties={"name": "Line 3", "line_code": "L3"}
            )
        ],
    ),
    # Printer Nodes
    NodeApply(
        space=SPACE,
        external_id="printer_11",
        sources=[
            NodeOrEdgeData(
                source=printer_view,
                properties={"name": "Printer 11", "printer_code": "P11"}
            )
        ],
    ),
    NodeApply(
        space=SPACE,
        external_id="printer_31",
        sources=[
            NodeOrEdgeData(
                source=printer_view,
                properties={"name": "Printer 31", "printer_code": "P31"}
            )
        ],
    ),
    NodeApply(
        space=SPACE,
        external_id="printer_32",
        sources=[
            NodeOrEdgeData(
                source=printer_view,
                properties={"name": "Printer 32", "printer_code": "P32"}
            )
        ],
    ),
]

# ---------------------------------------------------------
# 2. CREATE SHIFT REPORT HEADER (Printer 11 - Day Shift)
# ---------------------------------------------------------
report_id = "report_p11_20261207_day"
nodes.append(
    NodeApply(
        space=SPACE,
        external_id=report_id,
        sources=[
            NodeOrEdgeData(
                source=report_view,
                properties={
                    "report_date": "12/7/2026",
                    "shift": "5AM-5PM",
                    "mechanic": "Unassigned",
                    "group": "Group A",
                    "label": "Day Shift",
                },
            )
        ],
    )
)

# ---------------------------------------------------------
# 3. CREATE ALL 12 HOURLY ENTRIES (From Spreadsheet Image)
# ---------------------------------------------------------
# Tuple format: (hour_interval, hourly_prod, cum_prod, hourly_rew, cum_rew, blow, eff, downtime)
hourly_spreadsheet_data = [
    ("5 a 6", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 60.0),
    ("6 a 7", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 60.0),
    ("7 a 8", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 60.0),
    ("8 a 9", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 60.0),
    ("9 a 10", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 60.0),
    ("10 a 11", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 60.0),
    ("11 a 12", 1001.0, 1001.0, 1.0, 1001.0, 0.0, 0.01516667, 59.09),
    ("12 a 1", 6006.0, 7007.0, 6.0, 7007.0, 0.0, 0.091, 54.54),
    ("1 a 2", 0.0, 7007.0, 0.0, 7007.0, 0.0, 0.0, 60.0),
    ("2 a 3", 0.0, 7007.0, 0.0, 7007.0, 0.0, 0.0, 60.0),
    ("3 a 4", 4004.0, 11011.0, 4.0, 11011.0, 0.0, 0.06066667, 56.36),
    ("4 a 5", 2002.0, 13013.0, 2.0, 13013.0, 0.0, 0.03033333, 58.18),
]

edges = []

# Asset Hierarchy Edges
edges.extend([
    # Plant -> Lines
    EdgeApply(
        space=SPACE,
        external_id="plant_has_l1",
        type=DirectRelationReference(space=SPACE, external_id="hasLine"),
        start_node=DirectRelationReference(space=SPACE, external_id="plant_main"),
        end_node=DirectRelationReference(space=SPACE, external_id="line_l1"),
    ),
    EdgeApply(
        space=SPACE,
        external_id="plant_has_l3",
        type=DirectRelationReference(space=SPACE, external_id="hasLine"),
        start_node=DirectRelationReference(space=SPACE, external_id="plant_main"),
        end_node=DirectRelationReference(space=SPACE, external_id="line_l3"),
    ),
    # Lines -> Printers
    EdgeApply(
        space=SPACE,
        external_id="l1_has_p11",
        type=DirectRelationReference(space=SPACE, external_id="hasPrinter"),
        start_node=DirectRelationReference(space=SPACE, external_id="line_l1"),
        end_node=DirectRelationReference(space=SPACE, external_id="printer_11"),
    ),
    EdgeApply(
        space=SPACE,
        external_id="l3_has_p31",
        type=DirectRelationReference(space=SPACE, external_id="hasPrinter"),
        start_node=DirectRelationReference(space=SPACE, external_id="line_l3"),
        end_node=DirectRelationReference(space=SPACE, external_id="printer_31"),
    ),
    EdgeApply(
        space=SPACE,
        external_id="l3_has_p32",
        type=DirectRelationReference(space=SPACE, external_id="hasPrinter"),
        start_node=DirectRelationReference(space=SPACE, external_id="line_l3"),
        end_node=DirectRelationReference(space=SPACE, external_id="printer_32"),
    ),
    # Printer 11 -> Report
    EdgeApply(
        space=SPACE,
        external_id="p11_has_report_20261207_day",
        type=DirectRelationReference(space=SPACE, external_id="hasReport"),
        start_node=DirectRelationReference(space=SPACE, external_id="printer_11"),
        end_node=DirectRelationReference(space=SPACE, external_id=report_id),
    ),
])

# Process hourly rows into Entry Nodes + Edges from Report
for idx, (interval, prod, cum_prod, rew, cum_rew, blow, eff, dt) in enumerate(hourly_spreadsheet_data):
    entry_id = f"{report_id}_entry_{idx+1:02d}"
    
    # Entry Node
    nodes.append(
        NodeApply(
            space=SPACE,
            external_id=entry_id,
            sources=[
                NodeOrEdgeData(
                    source=entry_view,
                    properties={
                        "hour_interval": interval,
                        "hourly_production": prod,
                        "cumulative_production": cum_prod,
                        "hourly_rework": rew,
                        "cumulative_rework": cum_rew,
                        "blow_off": blow,
                        "efficiency": eff,
                        "downtime_minutes": dt,
                        "observations": "",
                    },
                )
            ],
        )
    )
    # Edge: Report -> Entry
    edges.append(
        EdgeApply(
            space=SPACE,
            external_id=f"{report_id}_has_{entry_id}",
            type=DirectRelationReference(space=SPACE, external_id="hasEntry"),
            start_node=DirectRelationReference(space=SPACE, external_id=report_id),
            end_node=DirectRelationReference(space=SPACE, external_id=entry_id),
        )
    )

# ---------------------------------------------------------
# 4. PUBLISH NODES AND EDGES TO CDF
# ---------------------------------------------------------
# Pass both lists directly to instances.apply()
client.data_modeling.instances.apply(nodes=nodes, edges=edges)

print(f"Population Complete! Applied {len(nodes)} Nodes and {len(edges)} Edges.")

Population Complete! Applied 19 Nodes and 18 Edges.


In [ ]:
from cognite.client.data_classes.data_modeling import (
    ContainerApply,
    ContainerProperty,
    ViewApply,
    MappedPropertyApply,
    ContainerId,
    DataModelApply,
    ViewId,
    Text,
    Float64,
)

SPACE = "TestSpace"
MODEL_NAME = "PlantProductionModel"
MODEL_VERSION_V2 = "v2"

# ---------------------------------------------------------------------------
# 1. Monkeypatch SDK to strip read-only 'constraintState' automatically
# ---------------------------------------------------------------------------
_orig_dump = ContainerProperty.dump

def _clean_dump(self, camel_case: bool = False):
    res = _orig_dump(self, camel_case=camel_case)
    if isinstance(res, dict):
        res.pop("constraintState", None)
        res.pop("constraint_state", None)
    return res

ContainerProperty.dump = _clean_dump


# ---------------------------------------------------------------------------
# 2. Update Container (Physical Storage)
# ---------------------------------------------------------------------------
entry_container = ContainerApply(
    space=SPACE,
    external_id="ProductionEntryContainer",
    name="Hourly Production Entry Storage",
    properties={
        "hour_interval": ContainerProperty(type=Text()),
        "hourly_production": ContainerProperty(type=Float64()),
        "cumulative_production": ContainerProperty(type=Float64()),
        "hourly_rework": ContainerProperty(type=Float64()),        # Kept for v1 compatibility
        "cumulative_rework": ContainerProperty(type=Float64()),    # Kept for v1 compatibility
        "hourly_retrac": ContainerProperty(type=Float64()),        # New for v2
        "cumulative_retrac": ContainerProperty(type=Float64()),    # New for v2
        "blow_off": ContainerProperty(type=Float64()),
        "efficiency": ContainerProperty(type=Float64()),
        "downtime_minutes": ContainerProperty(type=Float64()),
        "observations": ContainerProperty(type=Text()),
    },
)

# Native SDK apply now works seamlessly with the patch in place
client.data_modeling.containers.apply([entry_container])
print("ProductionEntryContainer successfully updated with retrac properties!")

# 2. Publish ProductionEntry View Version 2 (v2)
entry_view_v2 = ViewApply(
    space=SPACE,
    external_id="ProductionEntry",
    version="v2",
    name="Production Entry View V2",
    properties={
        "hour_interval": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="hour_interval",
        ),
        "hourly_production": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="hourly_production",
        ),
        "cumulative_production": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="cumulative_production",
        ),
        # Mapped to the new retrac container properties
        "hourly_retrac": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="hourly_retrac",
        ),
        "cumulative_retrac": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="cumulative_retrac",
        ),
        "blow_off": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="blow_off",
        ),
        "efficiency": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="efficiency",
        ),
        "downtime_minutes": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="downtime_minutes",
        ),
        "observations": MappedPropertyApply(
            container=ContainerId(space=SPACE, external_id="ProductionEntryContainer"),
            container_property_identifier="observations",
        ),
    },
)

client.data_modeling.views.apply([entry_view_v2])
print("View ProductionEntry/v2 successfully created!")

from cognite.client.data_classes.data_modeling import DataModelApply, ViewId

MODEL_NAME = "PlantProductionModel"

data_model_v2 = DataModelApply(
    space=SPACE,
    external_id=MODEL_NAME,
    version="v2",
    description="Manufacturing Plant Model v2 with updated Retrac fields",
    views=[
        ViewId(space=SPACE, external_id="Plant", version="v1"),
        ViewId(space=SPACE, external_id="Line", version="v1"),
        ViewId(space=SPACE, external_id="Printer", version="v1"),
        ViewId(space=SPACE, external_id="ProductionReport", version="v1"),
        ViewId(space=SPACE, external_id="ProductionEntry", version="v2"),  # Swapped to v2
    ],
)

client.data_modeling.data_models.apply([data_model_v2])
print("Data Model PlantProductionModel/v2 successfully published!")

CogniteNotFoundError:  | code: 404 | X-Request-ID: None | cluster: aw-was-gp-001 | project: alimentospolarcomercialca